# Q-Learning from Scratch: Navigating the Frozen Lake

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/q_learning_frozen_lake.ipynb)

This notebook accompanies the blog post at [sesen.ai](https://sesen.ai/blog/q-learning-frozen-lake-from-scratch).

We implement Q-learning from scratch to solve OpenAI's FrozenLake environment.

In [ ]:
!pip install -q gymnasium

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## The Environment

FrozenLake is a 4x4 grid: S=Start, F=Frozen (safe), H=Hole (game over), G=Goal (reward=1).
With `is_slippery=True`, actions have only 1/3 chance of going in the intended direction.

In [ ]:
env = gym.make('FrozenLake-v1', is_slippery=True, render_mode='ansi')
print(env.reset()[0])
print(env.render())
print(f"States: {env.observation_space.n}, Actions: {env.action_space.n}")
print("Actions: 0=Left, 1=Down, 2=Right, 3=Up")

## Q-Learning Implementation

In [ ]:
def q_learning(env, n_episodes=20000, alpha=0.1, gamma=0.99,
               epsilon_start=1.0, epsilon_end=0.01, decay_rate=1e-3,
               snapshot_episodes=None):
    """Q-learning with epsilon-greedy exploration."""
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    rewards = []
    snapshots = []

    epsilon = epsilon_start
    for episode in range(n_episodes + 1):
        if snapshot_episodes and episode in snapshot_episodes:
            snapshots.append((episode, Q.copy()))

        state, _ = env.reset()
        total_reward = 0

        for step in range(100):
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(Q[state])

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # Q-learning update (Bellman equation)
            Q[state, action] += alpha * (
                reward + gamma * np.max(Q[next_state]) - Q[state, action]
            )

            total_reward += reward
            state = next_state
            if done:
                break

        rewards.append(total_reward)
        epsilon = epsilon_end + (epsilon_start - epsilon_end) * np.exp(
            -decay_rate * episode
        )

    return Q, rewards, snapshots

In [ ]:
np.random.seed(42)
env = gym.make('FrozenLake-v1', is_slippery=True)

snap_eps = [0, 100, 500, 1000, 2000, 5000, 10000, 15000, 20000]
Q, rewards, snapshots = q_learning(env, snapshot_episodes=snap_eps)

# Show learned policy
actions = ['\u2190', '\u2193', '\u2192', '\u2191']
policy = np.array([actions[a] for a in np.argmax(Q, axis=1)]).reshape(4, 4)
print("Learned policy:")
print(policy)
print(f"\nSuccess rate (last 1000): {np.mean(rewards[-1000:]):.1%}")

## Learning Curve

In [ ]:
rolling = np.convolve(rewards, np.ones(100)/100, mode='valid')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rolling, 'b-', linewidth=0.8)
ax.set_xlabel('Episode')
ax.set_ylabel('Success Rate (100-episode rolling avg)')
ax.set_title('Q-Learning on FrozenLake (slippery)')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.show()

## Policy Evolution Animation

In [ ]:
arrow_dx = {0: -0.25, 1: 0, 2: 0.25, 3: 0}
arrow_dy = {0: 0, 1: 0.25, 2: 0, 3: -0.25}
holes = [5, 7, 11, 12]
goal_state = 15

fig, ax = plt.subplots(figsize=(6, 6))

def update(frame_num):
    ax.clear()
    ep, q = snapshots[frame_num]
    V = np.max(q, axis=1).reshape(4, 4)
    pol = np.argmax(q, axis=1)
    
    ax.imshow(V, cmap='YlOrRd', vmin=0, vmax=max(np.max(V), 0.01), aspect='equal')
    for i in range(5):
        ax.axhline(i - 0.5, color='black', linewidth=1)
        ax.axvline(i - 0.5, color='black', linewidth=1)
    
    for s in range(16):
        r, c = divmod(s, 4)
        if s in holes:
            ax.text(c, r, 'H', ha='center', va='center', fontsize=16, fontweight='bold', color='blue')
        elif s == goal_state:
            ax.text(c, r, 'G', ha='center', va='center', fontsize=16, fontweight='bold', color='green')
        elif s == 0:
            ax.text(c, r, 'S', ha='center', va='center', fontsize=14, fontweight='bold', color='gray')
            if np.max(q[s]) > 0.001:
                a = pol[s]
                ax.annotate('', xy=(c + arrow_dx[a]*1.5, r + arrow_dy[a]*1.5),
                           xytext=(c, r), arrowprops=dict(arrowstyle='->', color='black', lw=2))
        else:
            if np.max(q[s]) > 0.001:
                a = pol[s]
                ax.annotate('', xy=(c + arrow_dx[a]*1.5, r + arrow_dy[a]*1.5),
                           xytext=(c, r), arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(3.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Episode {ep:,}  |  V(s) = max Q(s,a)', fontsize=13)

anim = FuncAnimation(fig, update, frames=len(snapshots), interval=500)
HTML(anim.to_jshtml())

## Slippery vs Non-Slippery Comparison

In [ ]:
np.random.seed(42)
env_easy = gym.make('FrozenLake-v1', is_slippery=False)
Q_easy, rewards_easy, _ = q_learning(env_easy)
print(f"Non-slippery success rate (last 1000): {np.mean(rewards_easy[-1000:]):.1%}")

np.random.seed(42)
env_hard = gym.make('FrozenLake-v1', is_slippery=True)
Q_hard, rewards_hard, _ = q_learning(env_hard)
print(f"Slippery success rate (last 1000): {np.mean(rewards_hard[-1000:]):.1%}")

fig, ax = plt.subplots(figsize=(8, 4))
roll_easy = np.convolve(rewards_easy, np.ones(100)/100, mode='valid')
roll_hard = np.convolve(rewards_hard, np.ones(100)/100, mode='valid')
ax.plot(roll_easy, 'g-', linewidth=0.8, label='Non-slippery')
ax.plot(roll_hard, 'b-', linewidth=0.8, label='Slippery')
ax.set_xlabel('Episode')
ax.set_ylabel('Success Rate (100-ep rolling avg)')
ax.set_title('Slippery vs Non-Slippery FrozenLake')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Q-Table Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(Q, cmap='YlOrRd', aspect='auto')
ax.set_xlabel('Action (0=L, 1=D, 2=R, 3=U)')
ax.set_ylabel('State')
ax.set_title('Learned Q-Table')
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(['Left', 'Down', 'Right', 'Up'])
plt.colorbar(im, ax=ax)
plt.show()

## Exercises

1. **Slippery vs non-slippery** — Compare convergence curves. How does the optimal success rate differ?
2. **Gamma sweep** — Try gamma in {0.5, 0.8, 0.95, 0.99}. How does the discount factor affect learning?
3. **SARSA comparison** — Modify the update to use the actual next action instead of max. How does the policy differ?
4. **8x8 FrozenLake** — Switch to `FrozenLake8x8-v1`. Does Q-learning still converge?
5. **Visualise exploration** — Track visit counts per state. Which states are under-explored?